# DatensEE Demo — Export + Visualize

Run the built-in NDVI demo, export tiles to GCS, then visualize the result
on an interactive map using Earth Engine's `loadGeoTIFF` + geemap.

> **Private repo:** the Open in Colab badge doesn't work for private repos.
> Open this notebook via **File → Open notebook → GitHub**, connect your GitHub
> account, search for `michaelfdewitt/datensee`, and select this file.
> See the [collaborator quick start](https://github.com/michaelfdewitt/datensee#collaborator-quick-start-private-beta) in the README for full setup steps.

## 0. Setup

**Before running:** add a GitHub personal access token (with `repo` scope) to
Colab Secrets as `GITHUB_TOKEN` (the key icon in the left sidebar).
This is required while the repo is private.

This cell handles install, auth, and JAR download in one shot.

In [ ]:
# -- GitHub token (required for private repo access) --
import subprocess
from google.colab import userdata
_gh_token = userdata.get("GITHUB_TOKEN")

# Configure git to inject the token for github.com URLs.
# This covers both the pip install and any subsequent git operations.
subprocess.check_call([
    "git", "config", "--global",
    f"url.https://{_gh_token}@github.com/.insteadOf",
    "https://github.com/",
])

# -- Install --
!pip install -q "datensee @ git+https://github.com/michaelfdewitt/datensee.git#subdirectory=cli"
!pip install -q geemap

# -- Auth --
from datensee import notebook
notebook.ensure_auth()

# -- JAR --
jar_path = notebook.ensure_jar()
print(f"JAR: {jar_path}")

# -- Initialize EE (for geemap visualization later) --
import ee
PROJECT = "datensee-testing"  # <-- change this
ee.Initialize(project=PROJECT)

## 1. Configure

In [ ]:
GCS_BUCKET = "datensee-testing"  # <-- change this
OUTPUT = f"gs://{GCS_BUCKET}/demo-ndvi"

## 2. Export

In [ ]:
import datensee
from datensee.api import _demo_expression, _demo_region

result = datensee.export(
    ee_expression=_demo_expression(),
    region=_demo_region(),
    project=PROJECT,
    output=OUTPUT,
    runner="local",
    jar=jar_path,
)

print(f"Tiles OK: {result.tiles_ok}")
print(f"Duration: {result.duration_seconds:.1f}s")

## 3. Visualize with geemap

Build a server-side mosaic from the exported GeoTIFFs using
`ee.Image.loadGeoTIFF()`, then display it on an interactive map.

In [ ]:
import geemap

# Build an EE mosaic from the exported tiles on GCS
tiles = result.config.tile_grid.tiles
tile_images = [
    ee.Image.loadGeoTIFF(f"{OUTPUT}/tile_r{t.row:04d}_c{t.col:04d}.tif")
    for t in tiles
]
mosaic = ee.ImageCollection(tile_images).mosaic()

# Center the map on the export region
region = _demo_region()
coords = region["coordinates"][0]
center_lat = sum(c[1] for c in coords) / len(coords)
center_lon = sum(c[0] for c in coords) / len(coords)

m = geemap.Map(center=[center_lat, center_lon], zoom=11)
m.add_layer(
    mosaic,
    {"min": -0.2, "max": 0.8, "palette": ["brown", "lightyellow", "darkgreen"]},
    "NDVI",
)
m

## 4. Cleanup (optional)

Delete the exported tiles from GCS.

In [ ]:
# from google.cloud import storage
# client = storage.Client(project=PROJECT)
# prefix = OUTPUT.replace(f"gs://{GCS_BUCKET}/", "")
# blobs = list(client.list_blobs(GCS_BUCKET, prefix=prefix))
# for blob in blobs:
#     blob.delete()
# print(f"Deleted {len(blobs)} objects")